# 적용하기 — 보도자료 → 기사 첫 문장(리드) 학습 데이터
1. 수집 `press_raw.csv` → 2. 정제 `press_clean.csv` → 3. 리드 초안 `pairs_draft.csv` → 4. `lead_draft.jsonl`

예제 사이트: 문화체육관광부 보도자료 (공공누리 제1유형). 목록은 HTML, **본문은 첨부 PDF**에 있어요.

⚠️ 수집 전: robots.txt · 이용약관 · 공공누리 표시 확인. 받은 PDF·CSV·JSONL은 GitHub에 올리지 않아요.

* LIST_URL = 'https://www.mcst.go.kr/site/s_notice/press/pressList.jsp?pCurrentPage={}'

In [10]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

실행 위치: 내 컴퓨터 (Windows)
GPU: NVIDIA GeForce RTX 5070 Laptop GPU, 8151 MiB
한글 글꼴: ['Malgun Gothic']


In [11]:
# 필요한 패키지 확인 — 없으면 설치 (처음 한 번)
import importlib, subprocess, sys
for pkg, mod in [('pypdf', 'pypdf'), ('python-dotenv', 'dotenv'), ('openai', 'openai')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('패키지 준비 완료')

패키지 준비 완료


In [12]:
# ▶ API 키 불러오기 — Colab은 🔑 보안 비밀, VS Code는 같은 폴더의 .env 파일
#   .env 파일 내용 예:  OPENAI_API_KEY=sk-...   (GitHub에 올리지 않기)
def get_key(name='OPENAI_API_KEY'):
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        print('pip install python-dotenv 후 다시 실행')
    return os.getenv(name)

OPENAI_API_KEY = get_key()
print('API 키:', '준비됨' if OPENAI_API_KEY else '없음 → 키를 등록하세요')

API 키: 없음 → 키를 등록하세요


## 1. 수집 ① 목록에서 제목·링크 모으기
`a[href*='pressView.jsp']` = 주소에 `pressView.jsp`가 들어간 링크 = 보도자료 상세 페이지

In [13]:
import requests, time, re, io, pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

LIST_URL = 'https://www.mcst.go.kr/site/s_notice/press/pressList.jsp?pCurrentPage={}'
LINK = "a[href*='pressView.jsp']"
H = {'User-Agent': 'Mozilla/5.0'}

rows = []
for page in range(1, 2):                      # 먼저 1페이지만 → 잘 되면 range(1, 6) 등으로 늘리기
    soup = BeautifulSoup(requests.get(LIST_URL.format(page), headers=H).text, 'html.parser')
    for a in soup.select(LINK):
        rows.append({'제목': a.get_text(strip=True).replace('새글', '').strip(),
                     '링크': urljoin(LIST_URL, a['href'])})
    time.sleep(1)
print(len(rows), '건'); rows[:3]

10 건


[{'제목': '대중문화예술 작품 뒤 숨은 주역 10명(팀),  무대 위 주인공으로',
  '링크': 'https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22844'},
 {'제목': '문화진흥 4개 기관 통합 추진으로  국민 문화 향유 혁신 기반 마련한다',
  '링크': 'https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22843'},
 {'제목': '공개토론회로 「관광진흥법」 시행규칙 개정안 의견 수렴',
  '링크': 'https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22842'}]

## 1. 수집 ② 본문은 첨부 PDF에서
상세 페이지 HTML에는 본문이 없어요 (F12 → 「내려받기」 버튼 = `file_download('파일이름', '저장이름', '폴더번호')`).
그 세 값으로 PDF를 직접 받아 글자를 뽑아요. 받은 PDF는 `press_pdf` 폴더에 저장돼요.

In [14]:
from pathlib import Path
from pypdf import PdfReader

PDF_DIR = Path('press_pdf'); PDF_DIR.mkdir(exist_ok=True)
DOWN = 'https://www.mcst.go.kr/servlets/eduport/front/upload/UplDownloadFile'

def press_body(view_url):
    page = requests.get(view_url, headers=H).text
    m = re.search(r"file_download\('([^']+?\.pdf)',\s*'([^']+?\.pdf)',\s*'(\d+)'", page)
    if not m:
        print('PDF 첨부 없음:', view_url); return None
    name, saved, path = m.groups()
    url = f'{DOWN}?pFileName={name}&pRealName={saved}&pPath={path}&pFlag='
    r = requests.get(url, headers={**H, 'Referer': view_url})
    if not r.content.startswith(b'%PDF'):
        print('PDF를 못 받음:', r.status_code, r.headers.get('Content-Type')); return None
    (PDF_DIR / saved).write_bytes(r.content)
    return '\n'.join(p.extract_text() or '' for p in PdfReader(io.BytesIO(r.content)).pages)

print(press_body(rows[0]['링크'])[:600])       # 먼저 1개로 시험

- 1 -
문화체육관광부
 보도자료
보도시점 2026. 11. 12.(목) 08:00 배포 2026. 11. 12.(목) 08:00
대중문화예술 작품 뒤 숨은 주역 10명(팀), 
무대 위 주인공으로
- 11. 12. ‘2026 대중문화예술 제작스태프 대상’ 시상식 개최, 문체부 장관 
표창 4명, 콘진원장상 6명(팀) 수여
- 올해로 10회째, 연출·분장·무술·무대 등 작품 완성도 높인 제작진의 노고 격려
  문화체육관광부(장관 최휘영, 이하 문체부)는 한국콘텐츠진흥원(원장 
김윤지, 이하 콘진원)과 함께 11월 12일(목) 오후 5시, 스탠포드 호텔(서울 
마포구)에서 ‘2026 대중문화예술 제작스태프 대상’ 시상식을 열고 대중
문화예술 제작스태프 10명(팀)에게 상을 수여한다.
  ‘대중문화예술 제작스태프 대상’은 화면과 무대 뒤에서 작품을 완성해 
온 제작진을 발굴하고 그 노고를 격려하기 위해 2017년 처음 시상한 이래 
올해로 10회째를 맞았다.
 국민 추천과 전문가 심사로 4개 분야 제작진 10명(팀) 선정,
 <개그콘서트>부터 <기생충>까지, 명장면 뒤 숨은 손길에 문체부 장관 표창
  문체부와 콘진원은 5월부터 대중문화예술 분야에서 촬영, 조


In [15]:
for r in rows:                                 # 전체
    r['본문'] = press_body(r['링크'])
    time.sleep(1)
df = pd.DataFrame(rows)
df['본문'] = df['본문'].replace('', None)
print('본문 있는 글:', df['본문'].notna().sum(), '/', len(df))
assert df['본문'].notna().sum() > 0, '본문을 하나도 못 가져왔어요 → 위 칸의 메시지 확인'
df.to_csv('press_raw.csv', index=False, encoding='utf-8-sig')

본문 있는 글: 10 / 10


## 2. 정제 — 쪽 번호·머리말 지우기 · 빈 본문 · 중복 · 길이

In [16]:
df = pd.read_csv('press_raw.csv')
print('처음:', len(df))

df['본문'] = (df['본문']
             .str.replace(r'-\s*\d+\s*-', ' ', regex=True)                              # 쪽 번호 "- 1 -"
             .str.replace(r'보도시점.*?배포\s*[\d.\s]+\(.\)\s*[\d:]+', ' ', regex=True)   # 머리말
             .str.replace(r'\s+', ' ', regex=True).str.strip())
df = df.dropna(subset=['본문'])
df = df.drop_duplicates(subset=['본문'])
df = df[df['본문'].str.len() >= 300]           # 너무 짧은 글 빼기
df['본문'] = df['본문'].str[:2500]              # 긴 글은 앞 2,500자만

print('정제 후:', len(df))
assert len(df) > 0, '정제 후 0건 → 정제 조건이나 수집 결과 확인'
df.to_csv('press_clean.csv', index=False, encoding='utf-8-sig')
df['본문'].iloc[0][:300]

처음: 10
정제 후: 10


'문화체육관광부 보도자료 대중문화예술 작품 뒤 숨은 주역 10명(팀), 무대 위 주인공으로 - 11. 12. ‘2026 대중문화예술 제작스태프 대상’ 시상식 개최, 문체부 장관 표창 4명, 콘진원장상 6명(팀) 수여 - 올해로 10회째, 연출·분장·무술·무대 등 작품 완성도 높인 제작진의 노고 격려 문화체육관광부(장관 최휘영, 이하 문체부)는 한국콘텐츠진흥원(원장 김윤지, 이하 콘진원)과 함께 11월 12일(목) 오후 5시, 스탠포드 호텔(서울 마포구)에서 ‘2026 대중문화예술 제작스태프 대상’ 시상식을 열고 대중 문화예술 제작스태'

## 3. 리드 초안 — GPT가 쓰고, 확정은 사람이
키: VS Code는 레포 맨 위 폴더의 `.env` 파일에 `OPENAI_API_KEY=sk-...`

In [17]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)     # 위 폴더로 올라가며 .env 찾기
import os
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
print('API 키:', (OPENAI_API_KEY[:7] + '…') if OPENAI_API_KEY else '없음 → .env 확인')

API 키: 없음 → .env 확인


In [18]:
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

def draft_lead(body):
    prompt = f"""다음 보도자료로 기사 첫 문장(리드)을 한 문장으로 써.
- 보도자료에 없는 숫자·원인·전망은 쓰지 마.
- 누가·언제·무엇을 담아.
보도자료: {body[:2000]}"""
    r = client.chat.completions.create(model='gpt-4.1-mini',
                                       messages=[{'role': 'user', 'content': prompt}])
    return r.choices[0].message.content.strip()

df = pd.read_csv('press_clean.csv')
print(draft_lead(df['본문'].iloc[0]))          # 먼저 1개로 시험

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
df['리드'] = df['본문'].apply(draft_lead)       # 전체
df.to_csv('pairs_draft.csv', index=False, encoding='utf-8-sig')
df[['제목', '리드']].head()

,제목,리드
0,"‘제10회 서점의 날’, 책과 사람을 잇는 서점의 가치 되새겨",문화체육관광부 김영수 제1차관은 11월 11일 서울역사박물관에서 열린 ‘제10회 서...
1,"창립 20주년 한국도핑방지위원회, 새로운 도약 다짐",문화체육관광부는 11월 10일부터 11일까지 올림픽파크텔과 국립중앙도서관에서 한국도...
2,"청와대 사랑채, 강원 지역문화 상징하는 전시와 특별공연 선사",문화체육관광부는 11월 11일부터 12월 31일까지 청와대 사랑채에서 강원특별자치도...
3,"예술인의 세계 무대 도전 지원하는 ‘케이-아츠 온더고’, 2027년도 제1차 공모 시작",문화체육관광부는 11월 9일부터 23일까지 2027년과 2028년 개최 예정인 공연...
4,"한국문화를 처음 만난 세계인의 이야기, ‘토크토크 코리아 2026’ 수상작 124편 발표","문화체육관광부는 11월 9일, 외국인의 시선으로 한국문화를 조망하는 국제 콘텐츠 공..."


## 4. JSONL로 저장 — 한 줄 = 한 쌍 (alpaca 형식)

In [ ]:
import json
INST = '보도자료를 읽고 기사 첫 문장(리드)을 한 문장으로 쓰세요.'
with open('lead_draft.jsonl', 'w', encoding='utf-8') as f:
    for _, r in df.iterrows():
        row = {'instruction': INST, 'input': r['본문'], 'output': r['리드']}
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print(open('lead_draft.jsonl', encoding='utf-8').readline()[:200])

{"instruction": "보도자료를 읽고 기사 첫 문장(리드)을 한 문장으로 쓰세요.", "input": "문화체육관광부 보도자료 ‘제10회 서점의 날’, 책과 사람을 잇는 서점의 가치 되새겨 - 11. 11. 김영수 제1차관, ‘제10회 서점의 날’ 기념식에서 ‘인생독서×인생서점’ 우수 운영 ‘청학서림’ 등 서점 문화 발전 유공자 7명에게 장관 표창


다음 단계: `pairs_draft.csv`를 엑셀로 열어 **판정(사용·수정·제외)** 칸을 채우고 `pairs_checked.csv`로 저장